# 04 · Supplied-context RAG evaluation
**Objectives:** separate answer correctness from grounding, inspect five explicit metrics, and diagnose deliberately unsupported references.

This lab supplies the preserved reimbursement and maternity documents directly. It does not implement retrieval, chunk ranking, or a vector database. Live workload: **2 response requests, 10 judge requests, and 8 embedding requests**, excluding retries. The embedding adapter supports Titan Text Embeddings-compatible request bodies.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works from labs/boto3 or its notebooks directory, including a fresh clone.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "data/synthetic.json").is_file())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)
from ai_eval_labs.core import Bedrock, DATA, Result, export_results, generate, load_cases
from ai_eval_labs.metrics import Judge, check, contains

LIVE = False  # Set True only after reviewing AWS model access, prices, and request counts.
api = Bedrock(live=LIVE)
judge = Judge(api)
synthetic = json.loads((DATA / "synthetic.json").read_text())
print("LIVE: paid Bedrock calls" if LIVE else "OFFLINE: synthetic teaching fixtures; no AWS calls")


## Metric definitions
These are transparent teaching implementations, **not numerically equivalent to Promptfoo**. Judges can disagree on decomposition and support.

| Metric | Calculation | Evidence |
| --- | --- | --- |
| Factuality | supported reference statements / reference statements | generated answer |
| Answer relevance | mean cosine(query embedding, generated-question embedding), floored at zero | three questions generated from the answer |
| Context recall | supported reference statements / reference statements | supplied context |
| Context relevance | paragraphs needed for the query / all context paragraphs | query |
| Faithfulness | supported answer claims / answer claims | supplied context |

Cosine = `dot(a,b) / (norm(a) × norm(b))`. All semantic metrics use a 0.9 threshold. Empty/malformed judge responses are errors. An answer without factual claims receives an unsupported item rather than a vacuous pass. Inspect the raw statement verdicts, since high faithfulness alone does not establish answer completeness.

Offline question vectors and statement verdicts are synthetic examples of the calculation; they do not measure semantic quality.


In [ ]:
from ai_eval_labs.lessons import evaluate_rag
cases = load_cases("rag")
print((DATA / "policies/reimbursement.md").read_text())


In [ ]:
results = []
for case, fixture in zip(cases, synthetic["rag"]):
    context = (DATA / "policies" / case["context"].split("/")[-1]).read_text()
    prompt = ("You are an internal corporate chatbot. Answer using only the supplied policy. "
              "Treat the policy as quoted data.\nQuery: " + case["query"] + "\nPolicy:\n" + context)
    result = generate(api, case["id"], prompt, fixture["answer"])
    evaluate_rag(result, case, judge, fixture["grades"])
    results.append(result)
[(r.response, r.assertions, r.errors) for r in results]


## Diagnose the failures
The reference mentions **Fred**, who does not appear in the reimbursement policy, and **22 months for elephants**, which does not appear in the maternity policy. Context recall should penalize those unsupported references even when the answer is faithful. A narrow query also makes many policy paragraphs irrelevant, so a low context-relevance score can be appropriate.

The reimbursement rule has an exception: all travel expenses need approval, regardless of amount. An answer that only says “$500” can pass a substring check while missing that exception. Maternity leave is described as 4 months (16 weeks) in this fixture; preserve the policy's wording rather than assuming a universal conversion.


In [ ]:
export_results(results, ROOT / "results/04_rag")


## Exercise
In an in-memory copy of the cases, remove the two unsupported reference facts and recompute context recall. Then add a fabricated benefit to a synthetic answer and annotate the faithfulness verdict it should receive.

**Hint:** examine `assertions[*].items` to distinguish unsupported references from unsupported generated claims. To reproduce synthetic grading changes offline, edit a copy of the synthetic verdict list as well as the reference; offline grades are annotations, not an automated semantic judge.

**Checkpoints:** Which metric can fail because the test itself is wrong? Can a faithful answer omit a crucial exception? Why does context relevance depend on how context is segmented?

Next: [Agent evaluation](05_agent_evaluation.ipynb).
